In [ ]:
import stim

import sinter

from stimbposd import BPOSD, sinter_decoders

import numpy as np

import matplotlib.pyplot as plt

import multiprocessing

from pathlib import Path

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from circuit_library import memory as _circuit_library

_circuit_library.configure(5)

from circuit_library.memory import (
    stabilizers_k_z,
    stabilizers_k_x,
    stabilizers_general,
    DISTANCE,
    QUBIT_RANGE_3D,
    DEFAULT_SYNDROME_ANCILLA,
    DEFAULT_FLAG_QUBITS,
    LOGICAL_3D_PAULI,
    Z_WEIGHT_4_SEQUENCE,
    Z_WEIGHT_6_SEQUENCE,
    X_WEIGHT_8_SEQUENCE,
    X_WEIGHT_12_SEQUENCE,
    X_WEIGHT_18_SEQUENCE,
    MPP_CIRCUITS_BY_TYPE_AND_WEIGHT,
    MPP_circuit_X,
    MPP_circuit_Z,
    pauli_targets,
    pauli_string_type,
    pauli_weight,
    x_pauli_targets,
    z_pauli_targets,
    parse_flag_data_sequence,
    run_flag_data_sequence,
    MPP_circuit,
    count_measurements_for_stabilizers,
    Stabilizers_measurement_general,
    measure_logical_qubits_3D,
    MPP_circuit_X_weight_4,
    MPP_circuit_X_weight_6,
    MPP_circuit_X_weight_8,
    MPP_circuit_X_weight_12,
    MPP_circuit_X_weight_18,
    MPP_circuit_X_weight_24,
    MPP_circuit_Z_weight_4,
    MPP_circuit_Z_weight_6,
    MPP_circuit_Z_weight_8,
    MPP_circuit_Z_weight_12,
    MPP_circuit_Z_weight_18,
    MPP_circuit_Z_weight_24,
)


In [ ]:
def circuit_generate(rate_mea, rate_idle):
    data_qubit_range = list(range(0, 65))

    c = stim.Circuit()
    c.append("H", data_qubit_range)
    c += Stabilizers_measurement_general(0, 1)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 2)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 3)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 4)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 5)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle)
    c += Stabilizers_measurement_general(rate_mea, 6)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle)
    c += Stabilizers_measurement_general(0, 7)
    c.append("TICK")

    c += measure_logical_qubits_3D()
    return c


In [ ]:
c = circuit_generate(0.01, 0.01)
# c.diagram("timeline-svg")

In [ ]:
dem = c.detector_error_model()

In [ ]:
len(c.shortest_graphlike_error())

In [ ]:
def generate_tasks():
    for error_rate in [0.00005, 0.0002, 0.0004, 0.0006, 0.0008]:
        yield sinter.Task(
            circuit=circuit_generate(error_rate, error_rate),
            json_metadata={"p": error_rate},
        )


In [ ]:
samples = sinter.collect(
    num_workers=multiprocessing.cpu_count() - 1,
    max_shots=10_000_000,
    max_errors=100,
    tasks=generate_tasks(),
    decoders=["hypergraph_union_find"],
    custom_decoders=sinter_decoders(),
    save_resume_filepath="circuit_T_d5.csv"
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(  
    ax=ax,  
    stats=samples,  
    group_func=lambda stat: stat.decoder,  # No 'd' available  
    x_func=lambda stat: stat.json_metadata['p']  # Placeholder x since 'p' is unavailable  
)
ax.loglog()
ax.grid()
ax.set_title("Logical Error Rate vs Physical Error Rate")
ax.set_ylabel("Logical Error Probability (per shot)")
ax.set_xlabel("Physical Error Rate")
ax.legend()
plt.show()